# 03 — Add your own measure\n\nA measure is one decorated function; scoring it is one config line. We'll add **woodwork** (shots off the post) to the Chances family and rescore.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath("../src"))
os.environ.setdefault("EXCITEMENT_INDEX_CACHE", os.path.abspath("../.opendata_cache"))
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.width", 140)

In [2]:
from excitement_index.measures.registry import measure

@measure("woodwork")
def woodwork(ctx):
    """Shots that hit the post (excluding those saved onto it, which the
    goalkeeping family already credits)."""
    return float((ctx.shots["shot_outcome"] == "Post").sum())

New measures register at import time, so the feature build below picks `woodwork` up automatically. (Features must be rebuilt when the measure set changes — we rebuild just the knockouts here to keep it quick, then score the cached group-stage rows unchanged.)

In [3]:
from pathlib import Path
from excitement_index import opendata, build_feature_matrix

CACHE = Path("wc2022_features.parquet")
matches = opendata.load_matches("FIFA World Cup", "2022")
if CACHE.exists():
    features = pd.read_parquet(CACHE)
else:
    features = build_feature_matrix(matches, opendata.load_events,
                                    elo=opendata.load_elo(), jeopardy=False)
    features.to_parquet(CACHE)
print(features.shape)

(64, 64)


In [4]:
from excitement_index import extract_features, opendata

# recompute features for every match so the new column exists everywhere
elo = opendata.load_elo()
rows = {}
for mid in features.index:
    row = matches[matches["match_id"] == mid].iloc[0]
    rows[mid] = extract_features(opendata.load_events(int(mid)), row, elo=elo)
features2 = pd.DataFrame.from_dict(rows, orient="index")
for c in ("home", "away", "stage"):
    features2[c] = features[c]
features2["qualification_jeopardy"] = features["qualification_jeopardy"]
print("woodwork counts:", features2["woodwork"].sum())

woodwork counts: 33.0


## Score with the new measure in the taxonomy

In [5]:
from excitement_index import score_matches, load_config

cfg = load_config()
tax = {k: list(v) for k, v in cfg["taxonomy"].items()}
tax["chances"] = tax["chances"] + ["woodwork"]
board2 = score_matches(features2, config={"taxonomy": tax})
board2[["home", "away", "rating"]].head(10)

,home,away,rating
3869685,Argentina,France,9.59
3869420,Croatia,Brazil,9.49
3857292,Costa Rica,Germany,9.47
3869321,Netherlands,Argentina,9.38
3869354,England,France,9.29
3857284,Germany,Japan,9.27
3857259,Cameroon,Serbia,9.06
3869219,Japan,Croatia,9.04
3857256,Serbia,Switzerland,8.87
3857262,South Korea,Portugal,8.70
